In [1]:
import sys
import os
import networkx as nx
from typing import Hashable, TypeAlias
from tqdm import tqdm
import gc


In [2]:
CLASSES_PATH = os.path.dirname(os.path.abspath('D:/Code/Classes/'))
if not (CLASSES_PATH in sys.path):
    sys.path.append(CLASSES_PATH)
from Classes.Files_Handler_Class import Files_Handler
from Classes.Bcolors_Class import Bcolors as bcolors
from Classes.Random_Walk import Random_Walk
from Classes.Generate_Embedings import Generate_Embedings
from Classes.Load_Graph import Load_Graph

In [3]:
Random_Walk = Random_Walk()
Gnerate_Embedings = Generate_Embedings()
Load_Graph = Load_Graph()

In [4]:
Node: TypeAlias = Hashable  # Alias for readability

In [5]:
files_handler_obj = Files_Handler()
networks_list = []
networks_root_dir = ''
if networks_root_dir == '':
    networks_root_dir = files_handler_obj.select_dir() + '/'
# networks_root_dir = 'D:/Masters thesis/Networks Dataset/Monoplex/Test/'
networks_files = files_handler_obj.get_files_by_extensions(networks_root_dir, ['.edgeslist', '.edges'])


In [6]:
networks_files

['D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Kaktovi/Kaktovi.edgeslist',
 'D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Venetie/Venetie.edgeslist',
 'D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Wainwright/Wainwright.edges']

In [7]:
task_type = "IM"
rw_type = 'Vector'
rw_method = "Frequent Nodes"
embedding_attribute = 'Degree'
compression_method = 'lzma'
if embedding_attribute == 'Label' and rw_method == 'Scale':
    sys.exit(0)
if rw_method == "Frequent Nodes":
    rw_length = 512

In [8]:
final_rw_length = 512
embeddings_dimension = 128
walk_depth = 3
if rw_type == 'Vector':
    num_walks = 1
else:
    num_walks = 64
window = 3
min_count = 1
workers = 4
epochs = 100
batch_words = 4
p, q = 1, 0.5,

In [9]:
networks_files

['D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Kaktovi/Kaktovi.edgeslist',
 'D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Venetie/Venetie.edgeslist',
 'D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Wainwright/Wainwright.edges']

In [10]:
networks_info = {}
networks_random_walks = {}
accepable_files = [".edgeslist", ".edges"]
print(bcolors.WARNING + f"Networks load as a list of Geaphs.\n" + bcolors.ENDC)
graphs_of_networks_with_random_walk = {}
graphs_of_networks_without_random_walk = {}
i = 1
for item in networks_files:
    file_info = files_handler_obj.get_file_path_info(item)
    networks_info[file_info['name']] = file_info
    if file_info["type"] in accepable_files:
        print(f"{i}- {file_info['name']}")
        graphs_of_network, _, _, _ = Load_Graph.load_multilayer_graph(item, '\t', False)
        
        random_walk_load_status, networks_random_walks[file_info['name']] = Random_Walk.load_multilayer_network_nodes_random_walk(file_info, rw_method, rw_type,
                                                              final_rw_length, walk_depth, num_walks,
                                                              compression_method, tabs='\t')
        if random_walk_load_status != False:
            graphs_of_networks_with_random_walk[file_info['name']] = graphs_of_network
        else :
            graphs_of_networks_without_random_walk[file_info['name']] = graphs_of_network

        i += 1

Networks load as a list of Geaphs.

1- Kaktovi
	Network with 163 nodes and 37 layers loaded successfully.
	Random walk file not found.
2- Venetie
	Network with 205 nodes and 43 layers loaded successfully.
	Random walk file not found.
3- Wainwright
	Network with 217 nodes and 36 layers loaded successfully.
	Random walk file not found.


In [11]:
graphs_of_networks_without_random_walk_count = len(graphs_of_networks_without_random_walk)
print(f"{bcolors.green_fg}{bcolors.bold}Graphs with random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_with_random_walk)}{bcolors.end_color}")
print(f"{bcolors.red_fg}{bcolors.bold}Graphs without random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_without_random_walk)}{bcolors.end_color}")


Graphs with random walk: 0
Graphs without random walk: 3


In [12]:
def sort_list_by_frequency(input_list:list):
    frequency_dict = {}
    for item in input_list:
        if item in frequency_dict:
            frequency_dict[item] += 1
        else:
            frequency_dict[item] = 1
    sorted_items = sorted(frequency_dict.items(), key=lambda x: x[1], reverse=True)
    sorted_list = [item[0] for item in sorted_items]
    return sorted_list

In [13]:
gc.collect()
i = 1
for networks_name, graphs_of_network in graphs_of_networks_without_random_walk.items():
    print(f"{i}- {networks_name}")
    if rw_type == 'Vector':
        networks_random_walks[networks_info[networks_name]['name']] = Random_Walk.multilayer_graph_random_walk_vector(graphs_of_network, rw_length, walk_depth, attribute='label')
    print(f"\tgraph random walk size: {len(networks_random_walks[networks_info[networks_name]['name']])}")

        # ----------------- Frequent Nodes --------------------------------
    pbar = tqdm(total=len(networks_random_walks[networks_info[networks_name]['name']]))
    pbar.set_description(f"\tExtract Frequent Nodes")
    pbar.unit = ' Node'
    pbar.colour = 'Blue'
    for node, node_random_walks in networks_random_walks[networks_info[networks_name]['name']].items():
        for layer, random_walk in node_random_walks.items():
            maim_randome_walk = [random_walk[0]] # For main node stay first element
            maim_randome_walk.extend(sort_list_by_frequency(random_walk))
            if len(maim_randome_walk) > final_rw_length:
                maim_randome_walk = maim_randome_walk[:final_rw_length]
            elif  len(maim_randome_walk) < final_rw_length:
                maim_randome_walk.extend(random_walk[1:(final_rw_length - len(maim_randome_walk)+1)])
        pbar.update(1)
    pbar.close()
    #------------------------------------------------------------------        
    
    Random_Walk.write_multilayer_network_nodes_random_walk(networks_random_walks[networks_info[networks_name]['name']], networks_info[networks_name],
                                rw_method, rw_type,
                                  final_rw_length, walk_depth, num_walks,
                                  compression_method, tabs='\t')

    i += 1

1- Kaktovi
	Create nodes random walk vector:
		walk_length: 512, walk_depth: 3


		Layer 36  : 100%|██████████| 37/37 [00:01<00:00, 36.02 Layer/s]


	graph random walk size: 163


	Extract Frequent Nodes: 100%|██████████| 163/163 [00:00<00:00, 1055.13 Node/s]


	Write Kaktovi nodes random walk in file using lzma compression method.
		Write don.
		File path: D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Kaktovi/Kaktovi/
		File name: Kaktovi nodes Frequent Nodes random walk Vector walk_length=512 walk_depth=3 num_walks=1.xz
2- Venetie
	Create nodes random walk vector:
		walk_length: 512, walk_depth: 3


...

	Write Venetie nodes random walk in file using lzma compression method.
		Write don.
		File path: D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Venetie/Venetie/
		File name: Venetie nodes Frequent Nodes random walk Vector walk_length=512 walk_depth=3 num_walks=1.xz
3- Wainwright
	Create nodes random walk vector:
		walk_length: 512, walk_depth: 3


		Layer 35  : 100%|██████████| 36/36 [00:01<00:00, 21.30 Layer/s]


	graph random walk size: 217


	Extract Frequent Nodes: 100%|██████████| 217/217 [00:00<00:00, 1091.75 Node/s]


	Write Wainwright nodes random walk in file using lzma compression method.
		Write don.
		File path: D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Wainwright/Wainwright/
		File name: Wainwright nodes Frequent Nodes random walk Vector walk_length=512 walk_depth=3 num_walks=1.xz


In [14]:
graphs_of_networks_without_random_walk_count = len(graphs_of_networks_without_random_walk)
for networks_name, _ in graphs_of_networks_without_random_walk.items():
    graphs_of_networks_with_random_walk[networks_name] = graphs_of_networks_without_random_walk[networks_name]
graphs_of_networks_without_random_walk = {}
print(f"{bcolors.green_fg}{bcolors.bold}Graphs with random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_with_random_walk)}{bcolors.end_color}")
print(f"{bcolors.red_fg}{bcolors.bold}Graphs without random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_without_random_walk)}{bcolors.end_color}")

networks_graphs = graphs_of_networks_with_random_walk

Graphs with random walk: 3
Graphs without random walk: 0


In [15]:
def node_convert_random_walk_to_attribute(graph: nx.Graph, random_walk:list, attribute: str):
    attribute_values = []
    if attribute == 'Degree':
        for node in random_walk:
            attribute_values.append(graph.degree[node])
    
    return attribute_values

def graph_convert_random_walk_to_attribute(graphs_of_network: list[nx.Graph],
                                           network_random_walks:dict, attribute: str):
    temp_network_random_walks = network_random_walks
    pbar = tqdm(total=len(network_random_walks))
    pbar.set_description(f"\tGenerate Embedding")
    pbar.unit = ' Node'
    pbar.colour = 'Green'
    for node, layers_embs in network_random_walks.items():
        for layer, emb in layers_embs.items():
            temp_network_random_walks[node][layer] = node_convert_random_walk_to_attribute(graphs_of_network[int(layer)],
                                                                                       network_random_walks[node][layer],
                                                                                       attribute)
        pbar.update(1)
    pbar.close()
    return temp_network_random_walks

In [16]:
# gc.collect()
i = 1
for networks_name, network_random_walks in networks_random_walks.items():
    print(f"{i}- {networks_name}:")
    graph_nodes_embedding = None 
    embedding_load_status, network_embeddings = Generate_Embedings.load_multilayer_network_nodes_embedding(
        networks_info[networks_name], rw_method, rw_type, embedding_attribute,
                                                embeddings_dimension, walk_depth, num_walks,
                                                compression_method, load_in_RAM=True, tabs='\t')
    
    # print(graph_nodes_embedding)
    if embedding_load_status == False:
        if rw_type =='Vector':
            network_embeddings = graph_convert_random_walk_to_attribute(networks_graphs[networks_name],
                                                                        network_random_walks,
                                                                        embedding_attribute)
        
            
        Generate_Embedings.write_multilayer_network_nodes_embedding(network_embeddings, networks_info[networks_name],
                                                                    rw_method, rw_type, embedding_attribute,
                                                                    embeddings_dimension, walk_depth, num_walks,
                                                                    compression_method, tabs='\t')
    i += 1
print(f"\n------------------------- Finished -------------------------")

1- Kaktovi:
	Embedding file not found.


	Generate Embedding: 100%|██████████| 163/163 [00:00<00:00, 601.72 Node/s]


	Write Kaktovi nodes embedding in file using lzma compression method.
	Write don.
	File path: D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Kaktovi/Kaktovi/
	File name: Kaktovi nodes Degree Frequent Nodes Vector wl=128, wd=3 nw=1.xz
2- Venetie:
	Embedding file not found.


	Generate Embedding: 100%|██████████| 205/205 [00:00<00:00, 779.09 Node/s]


	Write Venetie nodes embedding in file using lzma compression method.
	Write don.
	File path: D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Venetie/Venetie/
	File name: Venetie nodes Degree Frequent Nodes Vector wl=128, wd=3 nw=1.xz
3- Wainwright:
	Embedding file not found.


	Generate Embedding: 100%|██████████| 217/217 [00:00<00:00, 544.72 Node/s]


	Write Wainwright nodes embedding in file using lzma compression method.
	Write don.
	File path: D:/Datasets/IM/Multilayer/00 - General Forms/Alaska-master - Copy/Wainwright/Wainwright/
	File name: Wainwright nodes Degree Frequent Nodes Vector wl=128, wd=3 nw=1.xz

------------------------- Finished -------------------------


In [17]:
# os.system('shutdown -s')